In [ ]:
import importlib

import pandas as pd

import alpha_bo
importlib.reload(alpha_bo)

from alpha_bo import get_log_path, run_one_trial, recompute_scores_in_csv

# User / campaign settings
USER = "Angze"
UNIVERSE = "TOP3000"
BATCH = 1

# Work out the exact CSV file for this user- and universe-specific campaign.
log_path = get_log_path(user=USER, universe=UNIVERSE)
existing_trials = len(pd.read_csv(log_path)) if log_path.exists() else 0

# Optional one-time period-score migration preview.
# This does not change the CSV; it reports period score / bo_score coverage.
# recompute_scores_in_csv(log_path, dry_run=True)
# If the preview looks right, run without dry_run to fill period-score columns after creating a backup.
# recompute_scores_in_csv(log_path)

print("BO campaign")
print("-" * 40)
print(f"User:     {USER}")
print(f"Universe: {UNIVERSE}")
print(f"Batch size: {BATCH}")
print(f"CSV log:  {log_path}")
print(f"Existing completed trials: {existing_trials}")


In [ ]:
if log_path.exists():
    existing_df = pd.read_csv(log_path)
    print(f"\nResuming from existing data: {len(existing_df)} completed trial(s) found.")
    display(existing_df.tail())
else:
    print("\nNo existing CSV log found. Starting a new campaign.")


In [ ]:
# Suggest BATCH candidate(s), then collect metrics candidate-by-candidate.
# For each candidate, paste the TRAIN Aggregate Data block and type DONE.
# You can optionally add TEST / IS blocks when prompted.
# Completed candidates append immediately; skipped candidates do not write partial rows.
new_result = run_one_trial(user=USER, universe=UNIVERSE, batch=BATCH)

if isinstance(new_result, list):
    saved_count = len(new_result)
    if saved_count == 0:
        print("\nNo completed candidates were saved.")
    else:
        print(f"\nSaved {saved_count} completed candidate(s) to:")
        print(log_path)
        display(pd.DataFrame(new_result))
elif new_result is None:
    print("\nNo completed candidate was saved.")
else:
    print("\nSaved 1 completed candidate to:")
    print(log_path)
    display(pd.DataFrame([new_result]))
